Step 1: Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

# Set a fixed random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Convert hex to numeric if applicable
def convert_hex_to_numeric(value):
    if isinstance(value, str) and value.startswith('0x'):
        return int(value, 16)
    else:
        return value

# Create a 7x7 matrix from a row of data
def create_7x7_matrix(data_row):
    numeric_row = pd.to_numeric(data_row.apply(convert_hex_to_numeric), errors='coerce').fillna(0)
    num_elements_required = 49
    if len(numeric_row) < num_elements_required:
        numeric_row = np.pad(numeric_row, (0, num_elements_required - len(numeric_row)), 'constant')
    matrix = np.array(numeric_row).reshape(7, 7)
    return matrix

# Process the dataset to create 7x7 matrices
def process_dataset(dataset):
    matrices = []
    for _, row in dataset.iterrows():
        matrix = create_7x7_matrix(row)
        matrices.append(matrix)
    return matrices

# Load dataset
df = pd.read_csv('UNSW_2018_IoT_Botnet.csv')

# Drop irrelevant columns
irrelevant_columns = ['pkSeqID', 'saddr', 'sport', 'daddr', 'dport', 'ltime', 'seq', 'attack', 'subcategory']
df = df.drop(columns=irrelevant_columns)

# Identify non-numeric columns (flgs, proto, state)
non_numeric_columns = ['flgs', 'proto', 'state']

# Encode non-numeric columns
le = LabelEncoder()
for col in non_numeric_columns:
    df[col] = le.fit_transform(df[col])

# Encode target labels (category)
label_encoder = LabelEncoder()
df['category'] = label_encoder.fit_transform(df['category'])

# Separate features and target
df_features = df.drop(columns=['category'])
df_target = df['category']

# Ensure all features are numeric
df_features = df_features.apply(pd.to_numeric, errors='coerce').fillna(0)

# Specify the columns to scale
columns_to_scale = df_features.columns  # Scale all feature columns
scaler = MinMaxScaler()
df_features_scaled = scaler.fit_transform(df_features)

# Convert the scaled features back to a DataFrame to apply the 7x7 matrix transformation
df_features_scaled = pd.DataFrame(df_features_scaled)

# Process each row to create 7x7 matrices
matrices = process_dataset(df_features_scaled)

# Convert to a 3D numpy array
X = np.array(matrices)

# Encode the labels
y = label_encoder.fit_transform(df_target)

# Split data
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=SEED)
# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

Step 2: Building the PDAE Model

In [4]:
import torch
import torch.nn as nn

class DeepAutoencoder(nn.Module):
    def __init__(self, num_input_features, num_classes, dropout_rate=0.4):
        super(DeepAutoencoder, self).__init__()
        self.num_input_features = num_input_features

        # Standard Encoder
        self.encoder_standard = nn.Sequential(
            nn.Linear(num_input_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64)
        )

        # Adjust input size for Conv1d with dilation=3
        conv1d_output_size_dilated2 = (num_input_features - 2 * (3 - 1) - 1) + 1  # Adjusted for dilation=2

       
        # Dilated Encoder with dilation factor of 2
        self.encoder_dilated2 = nn.Sequential(
            nn.Linear(num_input_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, stride=1, dilation=2),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(60, 64)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_input_features)
        )

        # Dilated Decoder with adjusted input and output sizes for dilation=2
        self.decoder_dilated2 = nn.Sequential(
            nn.Linear(32, conv1d_output_size_dilated2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.ConvTranspose1d(1, 1, kernel_size=3, stride=1, dilation=2),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(conv1d_output_size_dilated2, num_input_features)
        )

        # Classifier
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.view(-1, self.num_input_features)
        encoded_standard = self.encoder_standard(x)
        encoded_dilated2 = self.encoder_dilated2(x.unsqueeze(1))
        encoded_combined = torch.cat((encoded_standard, encoded_dilated2), dim=1)
        decoded = self.decoder(encoded_combined)
        classification = self.classifier(encoded_combined)
        return decoded, classification
        


# Example usage
num_classes = len(label_encoder.classes_)  # Update this based on your dataset
num_input_features = 49  # Update this based on your dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepAutoencoder(num_input_features, num_classes).to(device)


Step 3: Training the Model

In [5]:
import torch.optim as optim  # Import the optim module

LEARNING_RATE = 0.01  # Adjusted learning rate
BATCH_SIZE = 64        # Adjusted batch size

criterion_reconstruction = nn.L1Loss()
criterion_classification = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_data = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_data = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)


num_epochs = 100



for epoch in range(num_epochs):
    model.train()
    running_loss_reconstruction = 0.0
    running_loss_classification = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        decoded, classification = model(inputs.view(-1, num_input_features))
        loss_reconstruction = criterion_reconstruction(decoded, inputs.view(-1, num_input_features))
        loss_classification = criterion_classification(classification, labels)
        loss = loss_reconstruction + loss_classification
        loss.backward()
        optimizer.step()

        running_loss_reconstruction += loss_reconstruction.item()
        running_loss_classification += loss_classification.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Reconstruction Loss: {running_loss_reconstruction / len(train_loader)}, Classification Loss: {running_loss_classification / len(train_loader)}')
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            decoded, outputs = model(inputs.view(-1, num_input_features)) # فرض بر این است که مدل دو خروجی دارد
            _, predicted = torch.max(outputs, 1) # 'outputs' به جای 'outputs.data' استفاده شده است
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    validation_accuracy = correct / total
    print(f'Epoch {epoch+1}, Validation Accuracy: {validation_accuracy:.4f}')


Epoch [1/100], Reconstruction Loss: 0.048271910555640146, Classification Loss: 0.27512251662350634
Epoch 1, Validation Accuracy: 0.9555
Epoch [2/100], Reconstruction Loss: 0.048020590372900226, Classification Loss: 0.19678763163765167
Epoch 2, Validation Accuracy: 0.9694
Epoch [3/100], Reconstruction Loss: 0.04804264848394126, Classification Loss: 0.1765749998837991
Epoch 3, Validation Accuracy: 0.9691
Epoch [4/100], Reconstruction Loss: 0.04801332114175285, Classification Loss: 0.16035525650186738
Epoch 4, Validation Accuracy: 0.9592
Epoch [5/100], Reconstruction Loss: 0.04799952488844782, Classification Loss: 0.15853908099044564
Epoch 5, Validation Accuracy: 0.9586
Epoch [6/100], Reconstruction Loss: 0.04797171434377546, Classification Loss: 0.146525727228892
Epoch 6, Validation Accuracy: 0.9726
Epoch [7/100], Reconstruction Loss: 0.047987418585819876, Classification Loss: 0.1434806334177014
Epoch 7, Validation Accuracy: 0.9623
Epoch [8/100], Reconstruction Loss: 0.04804052610932126,

Step 4: Evaluating the Model and Calculating Metrics

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


model.eval()
true_labels = []
predicted_labels = []

test_data = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.view(-1, num_input_features)
        inputs, labels = inputs.to(device), labels.to(device)
        decoded, classification = model(inputs)
        _, predicted = torch.max(classification, 1)
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predicted.cpu().tolist())

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=1)
recall = recall_score(true_labels, predicted_labels, average='weighted', zero_division=1)
f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=1)
conf_matrix = confusion_matrix(true_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", conf_matrix)

Accuracy: 0.9793650036458795
Precision: 0.9800672698108771
Recall: 0.9793650036458795
F1 Score: 0.9793703425514103
Confusion Matrix:
 [[74078  2837     0     0     0]
 [  182 65989     0     0     0]
 [    0     0    18     1     0]
 [    0     0     3  3628     0]
 [    5     0     0     0     0]]
